In [1]:
# importing necessary libraries 

import pandas as pd # to load data and analyse, manipulate and visualize the DataFrame
import numpy as np # to deal with arrays and numerical values
from lightgbm import LGBMRegressor  # lightgbm model for regression problem
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error  # to evaluate the model performance
from sklearn.model_selection import train_test_split  # to split a dataset into subdatasets
from sklearn.model_selection import cross_val_score #  to train and test a model over multiple folds of the dataset
import pickle # to save the trained model
import optuna # to optimize the hyperparameters

C:\Users\user\Desktop\Synology_Friday_backup\Updated DRL Heating training and deployment codes_20230818\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the data into DataFrame with pandas package

data = pd.read_excel('07_ODU1_5min_for_energy_prediction_model.xlsx')

In [3]:
# provides a dimension of the DataFrame 
data.shape

(1540, 15)

In [4]:
# showing the column labels of the DataFrame
data.columns

Index(['ST', 'IAT', 'OAT', 'Delta_IAT_OAT', 'Delta_ST_IAT', 'Delta_ST_OAT',
       'High_Pressure', 'Day_sin', 'Day_cos', 'Op_Mode_0', 'Op_Mode_1',
       'Year_sin', 'Year_cos', 'OTW', 'Energy'],
      dtype='object')

In [5]:
# visualizing the first 5 rows of the DataFrame 
data.head()

,ST,IAT,OAT,Delta_IAT_OAT,Delta_ST_IAT,Delta_ST_OAT,High_Pressure,Day_sin,Day_cos,Op_Mode_0,Op_Mode_1,Year_sin,Year_cos,OTW,Energy
0,28.000000,22.511832,-1.180000,23.691832,5.488168,29.180000,29.750000,0.806445,-0.591310,0,1,-0.179679,0.983725,-1.835570,8066.800000
1,28.000000,26.084443,-1.021429,27.105871,1.915557,29.021429,31.678571,0.793353,-0.608761,0,1,-0.179620,0.983736,-1.856887,9899.071429
2,28.000000,28.685431,-1.003448,29.688879,-0.685431,29.003448,30.306897,0.779884,-0.625923,0,1,-0.179562,0.983747,-1.863094,8246.103448
3,28.000000,29.927749,-0.956667,30.884415,-1.927749,28.956667,29.760000,0.766044,-0.642788,0,1,-0.179503,0.983757,-1.892156,7415.033333
4,28.037447,30.608186,-0.856667,31.464853,-2.570740,28.894113,29.923333,0.751840,-0.659346,0,1,-0.179444,0.983768,-1.864571,7450.300000


In [6]:
# normalizing the inputs to transform features to be on a similar scale of 0 to 1 range



def normalize_inputs(data):
    ST_min, ST_max = 18, 31
    IAT_min, IAT_max = 15, 35
    OAT_min, OAT_max = -20, 20
    OTW_min, OTW_max = -15, 10
    High_Pressure_min, High_Pressure_max = 5, 35
    Energy_min, Energy_max = 0, 15000
    
    data['ST'] = (data['ST'] - ST_min) / (ST_max - ST_min)
    data['IAT'] = (data['IAT'] - IAT_min) / (IAT_max - IAT_min)
    data['OAT'] = (data['OAT'] - OAT_min) / (OAT_max - OAT_min)
    data['OTW'] = (data['OTW'] - OTW_min) / (OTW_max - OTW_min)
    data['High_Pressure'] = (data['High_Pressure'] - High_Pressure_min) / (High_Pressure_max - High_Pressure_min)
    data['Energy'] = (data['Energy'] - Energy_min) / (Energy_max - Energy_min)
    
    
    return data

# Denormalizing the predicted outputs for recovering the original values

def inverse_normalization_output(outputs):
    Energy_min, Energy_max = 0, 15000
    outputs['Energy'] = outputs['Energy'] * (Energy_max - Energy_min) + Energy_min
    
    return outputs

In [7]:
# Normalized data

data = normalize_inputs(data)

In [8]:
# function to transform time series data into a supervised learning dataset (input(X) and output(y))
# The function takes four arguments:
# 1. Sequence of observations as a list or 2D NumPy array 
# 2. Number of lag observations as input (X) 
# 3. Number of observations as output (y) 
# 4. Boolean whether or not to drop rows with NaN values 

def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):     
	n_vars = 1 if type(data) is list else data.shape[1]
	df = pd.DataFrame(data)
	cols, names = list(), list()
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):    
		cols.append(df.shift(i))
		names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
	# forecast sequence (t, t+1, ... t+n)  
	for i in range(0, n_out):
		cols.append(df.shift(-i))
		if i == 0:
			names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
		else:
			names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
	# put it all together
	agg = pd.concat(cols, axis=1)
	agg.columns = names
	# drop rows with NaN values
	if dropnan:
		agg.dropna(inplace=True)
	return agg  # The function returns a pandas DataFrame of series framed for supervised learning

In [9]:
#  To convert the values of the series from int type to a float type with 32 bits in computer memory

values = data.values.astype('float32')
reframed = series_to_supervised(values,2, 1) # to use two values at the previous time step
reframed = reframed.values                   # to convert the DataFrame into an array

In [10]:
# Distinguish the input features and output 

features = reframed[:, :-1]
targets = reframed[:, -1]

In [11]:
# Split the data into 80 % of the train data and 20 % of the test data

train_x, test_x, train_y, test_y = train_test_split(features, targets, test_size=0.2, shuffle=True, random_state=123)

In [12]:
# to initilaize the start time
import time
start = time.time()

In [13]:
# hyperparameters tuning
# to define objective function to be optimized

def objective(trial):

    param = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 3000, step=20),
        'n_estimators': trial.suggest_categorical('n_estimators', [10, 30, 50, 100, 200, 500]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.001, 0.01, 0.1, 0.3, 0.008,0.009]),
        'max_depth': trial.suggest_categorical('max_depth', [3,5,7,9,11,13,15,17,19]),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda':trial.suggest_loguniform('reg_lambda', 0.5, 1.0),
    }
    
    # Returns a list of 5 scores
    cv_score_lr = cross_val_score(LGBMRegressor(**param), train_x, train_y, cv=5, scoring='r2')
    
    # Average of the R2 to provide a single value 
    mean_r2 = round(sum(cv_score_lr)/len(cv_score_lr), 3) 
    
    

    return mean_r2

In [14]:
# To optimize the defined objective with a set of trials

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000)
print('Number of finished trials:', len(study.trials))

[I 2023-02-01 22:07:55,704] A new study created in memory with name: no-name-04c27e3b-033f-48f9-9e66-b748863f8edc
[I 2023-02-01 22:07:56,775] Trial 0 finished with value: 0.906 and parameters: {'num_leaves': 2180, 'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 15, 'reg_alpha': 0.3552428196259554, 'reg_lambda': 0.9322121003812701}. Best is trial 0 with value: 0.906.
[I 2023-02-01 22:07:56,896] Trial 1 finished with value: 0.143 and parameters: {'num_leaves': 60, 'n_estimators': 10, 'learning_rate': 0.01, 'max_depth': 13, 'reg_alpha': 0.41779210017810303, 'reg_lambda': 0.7104782243241688}. Best is trial 0 with value: 0.906.
[I 2023-02-01 22:07:57,790] Trial 2 finished with value: 0.748 and parameters: {'num_leaves': 2240, 'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 15, 'reg_alpha': 8.848032563980613, 'reg_lambda': 0.8513913393811187}. Best is trial 0 with value: 0.906.
[I 2023-02-01 22:08:00,214] Trial 3 finished with value: 0.872 and parameters: {'num_leaves': 1000, '

[I 2023-02-01 22:08:37,542] Trial 30 finished with value: 0.915 and parameters: {'num_leaves': 120, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 9, 'reg_alpha': 0.25148696458434294, 'reg_lambda': 0.9262429295305631}. Best is trial 29 with value: 0.917.
[I 2023-02-01 22:08:38,620] Trial 31 finished with value: 0.916 and parameters: {'num_leaves': 200, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 19, 'reg_alpha': 0.32122873890427145, 'reg_lambda': 0.9279759816976739}. Best is trial 29 with value: 0.917.
[I 2023-02-01 22:08:39,320] Trial 32 finished with value: 0.906 and parameters: {'num_leaves': 240, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 19, 'reg_alpha': 0.7444584848095613, 'reg_lambda': 0.9571214097279703}. Best is trial 29 with value: 0.917.
[I 2023-02-01 22:08:39,944] Trial 33 finished with value: 0.861 and parameters: {'num_leaves': 660, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 19, 'reg_alpha': 2.7525200309862816, 'reg_lambda': 

[I 2023-02-01 22:09:45,846] Trial 60 finished with value: 0.92 and parameters: {'num_leaves': 800, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.10560882173372699, 'reg_lambda': 0.9057275347123247}. Best is trial 52 with value: 0.921.
[I 2023-02-01 22:09:48,991] Trial 61 finished with value: 0.92 and parameters: {'num_leaves': 760, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 7, 'reg_alpha': 0.09798493324097603, 'reg_lambda': 0.8472872833574696}. Best is trial 52 with value: 0.921.
[I 2023-02-01 22:09:52,269] Trial 62 finished with value: 0.918 and parameters: {'num_leaves': 820, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.09541068401668297, 'reg_lambda': 0.8454319618957888}. Best is trial 52 with value: 0.921.
[I 2023-02-01 22:09:55,963] Trial 63 finished with value: 0.92 and parameters: {'num_leaves': 1120, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 7, 'reg_alpha': 0.02223933641108589, 'reg_lambda': 0

[I 2023-02-01 22:11:34,865] Trial 90 finished with value: 0.532 and parameters: {'num_leaves': 1140, 'n_estimators': 500, 'learning_rate': 0.001, 'max_depth': 15, 'reg_alpha': 0.07294191774875866, 'reg_lambda': 0.826823871176538}. Best is trial 52 with value: 0.921.
[I 2023-02-01 22:11:38,366] Trial 91 finished with value: 0.918 and parameters: {'num_leaves': 540, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 9, 'reg_alpha': 0.06090719894934327, 'reg_lambda': 0.9654052909309655}. Best is trial 52 with value: 0.921.
[I 2023-02-01 22:11:42,022] Trial 92 finished with value: 0.922 and parameters: {'num_leaves': 780, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.08867294103757506, 'reg_lambda': 0.8464704573529813}. Best is trial 92 with value: 0.922.
[I 2023-02-01 22:11:46,804] Trial 93 finished with value: 0.92 and parameters: {'num_leaves': 1020, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 9, 'reg_alpha': 0.032350671938965576, 'reg_lambd

[I 2023-02-01 22:13:17,102] Trial 120 finished with value: 0.758 and parameters: {'num_leaves': 1440, 'n_estimators': 10, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.04327737673448954, 'reg_lambda': 0.911125289468065}. Best is trial 92 with value: 0.922.
[I 2023-02-01 22:13:20,476] Trial 121 finished with value: 0.92 and parameters: {'num_leaves': 700, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 9, 'reg_alpha': 0.09694741504908262, 'reg_lambda': 0.8463334601753938}. Best is trial 92 with value: 0.922.
[I 2023-02-01 22:13:24,508] Trial 122 finished with value: 0.918 and parameters: {'num_leaves': 1160, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 7, 'reg_alpha': 0.02382887684546764, 'reg_lambda': 0.9724064009808541}. Best is trial 92 with value: 0.922.
[I 2023-02-01 22:13:30,735] Trial 123 finished with value: 0.918 and parameters: {'num_leaves': 1100, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.05159871580898013, 'reg_lamb

...

[I 2023-02-01 23:38:40,632] Trial 900 finished with value: 0.916 and parameters: {'num_leaves': 1700, 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 15, 'reg_alpha': 0.04663226721236281, 'reg_lambda': 0.6886964511516421}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:38:46,023] Trial 901 finished with value: 0.921 and parameters: {'num_leaves': 2560, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 11, 'reg_alpha': 0.08834323717999126, 'reg_lambda': 0.9495031031778043}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:38:46,422] Trial 902 finished with value: 0.755 and parameters: {'num_leaves': 1300, 'n_estimators': 10, 'learning_rate': 0.1, 'max_depth': 17, 'reg_alpha': 0.08656031433034458, 'reg_lambda': 0.7342062068005847}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:38:51,100] Trial 903 finished with value: 0.921 and parameters: {'num_leaves': 2480, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 11, 'reg_alpha': 0.1097623676300221, 'r

[I 2023-02-01 23:41:16,044] Trial 930 finished with value: 0.922 and parameters: {'num_leaves': 1000, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.05059030839788376, 'reg_lambda': 0.851918543642554}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:41:20,378] Trial 931 finished with value: 0.92 and parameters: {'num_leaves': 2600, 'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.0730255016244315, 'reg_lambda': 0.8014048711177159}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:41:23,998] Trial 932 finished with value: 0.922 and parameters: {'num_leaves': 960, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 7, 'reg_alpha': 0.04840478061471928, 'reg_lambda': 0.8601930690357292}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:41:29,089] Trial 933 finished with value: 0.921 and parameters: {'num_leaves': 2620, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.11503222313687761, 'reg_l

[I 2023-02-01 23:43:35,141] Trial 960 finished with value: 0.92 and parameters: {'num_leaves': 940, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.07836607334933014, 'reg_lambda': 0.8915856422454588}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:43:39,415] Trial 961 finished with value: 0.921 and parameters: {'num_leaves': 520, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.07045157778754885, 'reg_lambda': 0.9297918908763413}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:43:40,965] Trial 962 finished with value: 0.916 and parameters: {'num_leaves': 700, 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.05158460348731626, 'reg_lambda': 0.7514239728505018}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:43:44,555] Trial 963 finished with value: 0.92 and parameters: {'num_leaves': 1480, 'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 11, 'reg_alpha': 0.07111718250068633, 'reg_l

[I 2023-02-01 23:45:50,927] Trial 990 finished with value: 0.92 and parameters: {'num_leaves': 2740, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.13117031566904347, 'reg_lambda': 0.9794761334644675}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:45:53,147] Trial 991 finished with value: 0.914 and parameters: {'num_leaves': 780, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 15, 'reg_alpha': 0.04135995826713577, 'reg_lambda': 0.8526403029136658}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:45:53,628] Trial 992 finished with value: 0.459 and parameters: {'num_leaves': 760, 'n_estimators': 50, 'learning_rate': 0.008, 'max_depth': 5, 'reg_alpha': 0.0584801501124371, 'reg_lambda': 0.8971865583383024}. Best is trial 178 with value: 0.923.
[I 2023-02-01 23:45:58,276] Trial 993 finished with value: 0.921 and parameters: {'num_leaves': 2680, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 11, 'reg_alpha': 0.11376049665289806, 'reg

Number of finished trials: 1000


In [15]:
# to print out the time taken for optimizing

end = time.time()
print(end - start)

5901.680029392242


In [16]:
# print out the best hyperparameters
print('Best trial:', study.best_trial.params)

Best trial: {'num_leaves': 2720, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 11, 'reg_alpha': 0.10751625873990098, 'reg_lambda': 0.7253144420019486}


In [17]:
# Fitting the model (LGBMRegressor) and evaluate model performance

param = {
    "num_leaves": 2720,
    "n_estimators": 500,
    'learning_rate': 0.1,
    'max_depth': 11,
    "reg_alpha": 0.10751625873990098,
    'reg_lambda':0.7253144420019486,
    
}


model = LGBMRegressor(**param)

model.fit(train_x,train_y, eval_set=[(test_x, test_y)], verbose=-1)

C:\Users\HuBEL\anaconda3\lib\site-packages\lightgbm\sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "


LGBMRegressor(max_depth=11, n_estimators=500, num_leaves=2720,
              reg_alpha=0.10751625873990098, reg_lambda=0.7253144420019486)

In [18]:
# Predictin on train dataset

yhat_train = model.predict(train_x)

predicted_train=pd.DataFrame(yhat_train,columns=['Energy'])
predicted_train = inverse_normalization_output(predicted_train) # to reverse the normalization process

actual_train = pd.DataFrame(train_y,columns=['Energy'])
actual_train = inverse_normalization_output(actual_train)

print('Training results')
print('r^2:',r2_score(actual_train, predicted_train))
print('RMSE:',np.sqrt(mean_squared_error(actual_train, predicted_train)))
print('MAE:',mean_absolute_error(actual_train, predicted_train))


Training results
r^2: 0.9972232565387956
RMSE: 115.20352651873273
MAE: 75.38816296446646


In [19]:
# Predicting on test dataset

yhat_test = model.predict(test_x)
predicted_test=pd.DataFrame(yhat_test,columns=['Energy'])
predicted_test = inverse_normalization_output(predicted_test) # to reverse the normalization process

actual_test=pd.DataFrame(test_y,columns=['Energy'])
actual_test = inverse_normalization_output(actual_test)

print('Testing results')
print('r^2:',r2_score(actual_test, predicted_test))
print('RMSE:',np.sqrt(mean_squared_error(actual_test, predicted_test)))
print('MAE:',mean_absolute_error(actual_test, predicted_test))

Testing results
r^2: 0.9217521403564632
RMSE: 611.1564230359071
MAE: 320.97593357534225


In [20]:
# to save the model for use at a later time or place

pickle.dump(model, open('Energy.pkl','wb'))

In [21]:
# To concate, save the actual and predicted values into excel file

train_df = pd.concat([actual_train, predicted_train], axis=1)
train_df.to_excel("training_results_past_2.xlsx",index=False)
test_df = pd.concat([actual_test, predicted_test], axis=1)
test_df.to_excel("testing_results_past_2.xlsx",index=False)